# 03 — Curriculum Validity Check

In [ ]:
import numpy as np, json
from pathlib import Path

# Point to a real output_dir if you have one, otherwise use synthetic
PHI_PATH = None  # e.g. "output/run1/influence_matrix.npy"
IDS_PATH = None  # e.g. "output/run1/doc_ids.json"

if PHI_PATH and Path(PHI_PATH).exists():
    Phi = np.load(PHI_PATH)
    doc_ids = json.loads(Path(IDS_PATH).read_text())
else:
    rng = np.random.default_rng(99)
    D, T = 500, 3
    Phi = rng.random((D, T)).astype(np.float32)
    doc_ids = [f"fake#{i}" for i in range(D)]
    print(f"Using synthetic Phi {Phi.shape}")

print("Phi shape:", Phi.shape)

In [ ]:
import json, tempfile
from influence_curriculum.curriculum import CurriculumConfig, build_curriculum

tmp = tempfile.mkdtemp()
texts = [f"placeholder text {i}" for i in range(len(doc_ids))]
build_curriculum(Phi, texts, doc_ids, CurriculumConfig(), tmp, seed=0)

D, T = Phi.shape
for e in range(T):
    path = Path(tmp) / "curriculum" / f"epoch_{e:02d}.jsonl"
    ids = [json.loads(l)["id"] for l in path.read_text().splitlines()]
    ok = sorted(ids) == sorted(doc_ids)
    print(f"epoch {e:02d}: {len(ids)} docs, is_permutation={ok}")

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

source_mix = []
for e in range(T):
    path = Path(tmp) / "curriculum" / f"epoch_{e:02d}.jsonl"
    ids = [json.loads(l)["id"] for l in path.read_text().splitlines()]
    counts = Counter(i.split("#")[0] for i in ids)
    source_mix.append(counts)

sources = sorted(source_mix[0].keys())
x = np.arange(T)
bottom = np.zeros(T)
for src in sources:
    vals = np.array([m.get(src, 0) for m in source_mix])
    plt.bar(x, vals / len(doc_ids), bottom=bottom, label=src)
    bottom += vals / len(doc_ids)

plt.xlabel("epoch"); plt.ylabel("fraction"); plt.legend(loc="upper right")
plt.title("Source mix per epoch"); plt.show()